# 🌿 SmartPlant50 — One-click dataset acquisition

This notebook **reuses the nine crop ZIP archives already in your Google Drive** and downloads eligible public dataset files from publisher APIs directly into your **private Drive quarantine**. It requires only a Google Colab runtime and your consent to mount Drive.

**What this notebook does:** scans archives, discovers public dataset file metadata, downloads only those with a recognized licence and provider SHA-256, checks their byte integrity, and writes a source inventory.

**What it does *not* do:** it does not automatically approve plant-disease labels, manufacture missing images, create training/test splits, start CNN training, change your Android app, or deploy anything. Some datasets may be skipped when their API has no public SHA-256 or a compatible licence.

Recommended network/storage cap: **4 GiB** per execution. Colab may require Google authorization to access Drive.

## 1 · Get the verified acquisition scripts (no personal access tokens)

This fetches only publicly available project source code and installs Python dependencies in your own Colab runtime. All files downloaded later stay in your Drive under `SmartPlant50`.

In [ ]:
import subprocess, sys
from pathlib import Path

ROOT = Path("/content")
PROJECT = ROOT / "smartplant50-source-import"
BRANCH = "feature/smartplant50-one-click-colab"
REPO = "https://github.com/dapperdude404/tinyguard-ai.git"

def run(*args, cwd=None):
    subprocess.run([str(arg) for arg in args], cwd=cwd, check=True)

if PROJECT.exists():
    assert (PROJECT / ".git").exists(), "Unexpected existing folder: restart Colab runtime"
    run("git", "fetch", "--depth", "1", "origin", BRANCH, cwd=PROJECT)
    run("git", "checkout", "-B", BRANCH, "FETCH_HEAD", cwd=PROJECT)
else:
    run("git", "clone", "--depth", "1", "--branch", BRANCH, REPO, PROJECT)
print("Source commit:", subprocess.check_output(
    ["git", "rev-parse", "HEAD"], cwd=PROJECT, text=True).strip())
run(sys.executable, "-m", "pip", "install", "-q", "requests>=2.32,<3", "pillow>=10,<13")

## 2 · Connect your Google Drive

Google will ask you to approve Drive access for your own Colab session. This cannot be authorized from GitHub or a ChatGPT message.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DRIVE = Path("/content/drive/MyDrive")
LEGACY = DRIVE / "SmartPlant_Datasets"
WORKSPACE = DRIVE / "SmartPlant50"
REPORTS = WORKSPACE / "audit_reports"
QUARANTINE = WORKSPACE / "source_quarantine"

assert DRIVE.is_dir(), "Google Drive did not mount"
assert LEGACY.is_dir(), "SmartPlant_Datasets folder not found in your Drive"
WORKSPACE.mkdir(parents=True, exist_ok=True)
REPORTS.mkdir(parents=True, exist_ok=True)
QUARANTINE.mkdir(parents=True, exist_ok=True)
print("Found existing ZIPs:", sorted(p.name for p in LEGACY.glob("*.zip")))
print("Private download destination:", QUARANTINE)

## 3 · Scan the nine existing crops (metadata only)

This step does not extract any images or change their labels. It inventories ZIP folders and writes reports in `SmartPlant50/audit_reports`.

In [ ]:
run(sys.executable, "scripts/scan_smartplant_drive_archives.py",
    "--source-dir", LEGACY, "--report-dir", REPORTS, cwd=PROJECT)
print("Existing crop ZIP source inventory is saved in Drive.")

## 4 · Discover publisher datasets and file checksums

The source register currently covers proposed Mango, Guava, Banana, Rice, Wheat, Citrus, Maize and Papaya categories. The downloader supports versioned public **Mendeley Data** sources; other sources (for example, Zenodo or Paddy Doctor) are flagged as not yet supported. No leaf photographs are downloaded in this step.

In [ ]:
REGISTER = PROJECT / "dataset_sources/50class_source_register.csv"
run(sys.executable, "scripts/acquire_smartplant50_sources.py",
    "--registry", REGISTER, "--workspace", WORKSPACE, cwd=PROJECT)
print("Publisher file discovery report:",
      REPORTS / "external_source_receipts.json")

## 5 · Download SHA-verified eligible files to your private Drive

User-approved automatic acquisition. Files are downloaded only when the publisher provides a recognised CC BY 4.0 licence, a versioned source, an acceptable file type and a SHA-256 checksum. All are saved under **source_quarantine**, not into training folders. Skipped items remain documented.

**Storage/network budget:** 4 GiB. This may consume Google Drive quota, mobile data, and Colab runtime bandwidth. Use Wi-Fi. You can change `MAX_GIB` to a smaller amount before running.

In [ ]:
MAX_GIB = 4.0
assert 0 < MAX_GIB <= 4.0, "Choose a max budget between 0 and 4 GiB"
run(sys.executable, "scripts/acquire_smartplant50_sources.py",
    "--registry", REGISTER, "--workspace", WORKSPACE,
    "--download", "--max-total-gib", MAX_GIB, cwd=PROJECT)
print("Downloaded files have been SHA-verified into Drive quarantine.")

## 6 · Verify the results and find what remains missing

The receipts distinguish downloaded, cached, skipped and unsupported source files. No training begins even if all downloads succeed.

In [ ]:
import json
report_file = REPORTS / "external_source_receipts.json"
assert report_file.is_file(), "No acquisition receipts found"
report = json.loads(report_file.read_text(encoding="utf-8"))
print("Dataset sources discovered:", len(report["source_datasets"]))
print("Files verified in quarantine:", report["verified_download_count"])
print("Newly downloaded bytes:", format(report["downloaded_bytes"], ","))
for item in report["source_datasets"]:
    print("\nDataset:", item.get("name") or item["dataset_id"],
          "| version", item["version"],
          "| CC BY 4.0 metadata:", item["cc_by_4_metadata_verified"])
    for file in item["files"]:
        status = file.get("acquisition", {})
        print(" •", file["filename"], status.get("status", "UNKNOWN"),
              status.get("reason", ""))
print("\nQuarantine files:", len(list(QUARANTINE.glob("*/*"))))
print("Review source licences, class labels and group IDs before training.")

## 7 · Next research gate (not executed here)

After downloads, the separate class review and plant/farm provenance workflow determines which images are eligible for the 50-class model. **Do not label these files as training-ready just because checksums passed.**

The original trained 29-class Android and Streamlit deployments remain unchanged. The 50-class model still needs independently reviewed images and proper train/validation/test partitions.